In [2]:
# ============================================================
# CELL 1: WEB3 SETUP AND ETHEREUM CONNECTION
# ============================================================
#
# DAY 4: Connect to Ethereum mainnet via Infura
#
# WHAT IS WEB3?
# ─────────────────────────────────────────────────────────────
# Web3 is the Python library for interacting with Ethereum
# blockchain. It allows us to:
# - Read smart contract data (Uniswap pools, reserves)
# - Query blockchain state (balances, prices)
# - Monitor transactions (in real-time)
#
# WHY INFURA?
# ─────────────────────────────────────────────────────────────
# Running your own Ethereum node requires:
# - Downloading 1TB+ of blockchain data
# - 24/7 server uptime
# - Technical maintenance
#
# Infura provides FREE access to Ethereum nodes
# You just need an API key (Project ID)
#
# IMPORTANT: We are READING data only (no gas fees!)
# We're not sending transactions, just querying information
# ============================================================

from web3 import Web3
import json
from datetime import datetime
import pandas as pd

print("=" * 70)
print("DAY 4: WEB3 SETUP & ETHEREUM CONNECTION")
print("=" * 70)
print()

# ─────────────────────────────────────────────────────────────
# STEP 1: CONNECT TO ETHEREUM MAINNET
# ─────────────────────────────────────────────────────────────
# Your Infura API Key (already filled in for you!)
# ─────────────────────────────────────────────────────────────

INFURA_API_KEY = '910cab7eff0f4e30a6a7c285dbd36b1d'

# Build Infura URL
# Format: https://mainnet.infura.io/v3/YOUR_API_KEY
infura_url = f'https://mainnet.infura.io/v3/{INFURA_API_KEY}'

# Create Web3 connection
# [CLASS] Web3.HTTPProvider creates connection to Ethereum node via HTTP
w3 = Web3(Web3.HTTPProvider(infura_url))

print("📡 Connecting to Ethereum mainnet via Infura...")
print(f"   API Key: {INFURA_API_KEY[:8]}...{INFURA_API_KEY[-8:]}")
print()

# ─────────────────────────────────────────────────────────────
# STEP 2: TEST CONNECTION
# ─────────────────────────────────────────────────────────────
# w3.is_connected() returns True if we can reach Ethereum
# This verifies our API key is valid and Infura is responding
# ─────────────────────────────────────────────────────────────

if w3.is_connected():
    print("✅ Successfully connected to Ethereum mainnet!")
    print()
    
    # Get latest block number
    # [METHOD] w3.eth.block_number returns the most recent block
    # Ethereum produces ~1 block every 12 seconds
    # Block number constantly increases (proof connection is live)
    latest_block = w3.eth.block_number
    print(f"📦 Latest block number: {latest_block:,}")
    
    # Get block timestamp to show current blockchain time
    # [METHOD] w3.eth.get_block() returns full block details
    # Timestamp is Unix time (seconds since Jan 1, 1970)
    block = w3.eth.get_block(latest_block)
    block_time = datetime.fromtimestamp(block['timestamp'])
    print(f"⏰ Latest block timestamp: {block_time}")
    print()
    
    # Get current gas price
    # [METHOD] w3.eth.gas_price returns gas price in Wei
    # Wei is smallest unit of ETH (1 ETH = 10^18 Wei)
    # We convert to Gwei (1 Gwei = 10^9 Wei) for readability
    gas_price_wei = w3.eth.gas_price
    gas_price_gwei = w3.from_wei(gas_price_wei, 'gwei')
    print(f"⛽ Current gas price: {gas_price_gwei:.2f} Gwei")
    
    # Calculate cost of a simple ETH transfer
    # Standard ETH transfer = 21,000 gas units
    # Cost = gas_units × gas_price
    eth_transfer_cost_wei = 21000 * gas_price_wei
    eth_transfer_cost_eth = w3.from_wei(eth_transfer_cost_wei, 'ether')
    print(f"💸 Cost of ETH transfer: {eth_transfer_cost_eth:.6f} ETH")
    print(f"   (21,000 gas × {gas_price_gwei:.2f} Gwei)")
    print()
    
    # Calculate cost of Uniswap swap (more complex, uses more gas)
    # Uniswap swap typically uses ~150,000 gas
    uniswap_swap_cost_wei = 150000 * gas_price_wei
    uniswap_swap_cost_eth = w3.from_wei(uniswap_swap_cost_wei, 'ether')
    print(f"💱 Cost of Uniswap swap: {uniswap_swap_cost_eth:.6f} ETH")
    print(f"   (150,000 gas × {gas_price_gwei:.2f} Gwei)")
    print()
    
    print("🌐 Blockchain connection established!")
    print("   You can now read any public data from Ethereum")
    print("   Including: prices, pool reserves, balances, etc.")
    print()
    
else:
    print("❌ Connection failed!")
    print("   Check your Infura API Key")
    print("   Verify you have internet connection")
    print()

print("=" * 70)
print("WHAT YOU CAN DO NOW:")
print("=" * 70)
print()
print("With this Web3 connection, you can:")
print("  • Read Uniswap pool reserves")
print("  • Query token balances")
print("  • Monitor liquidity changes")
print("  • Track impermanent loss")
print("  • Get live ETH prices from DEX")
print("  • All without paying gas fees!")
print()
print("✅ Ready for Cell 2: Query Uniswap V3 pool for live ETH/USDC data")

DAY 4: WEB3 SETUP & ETHEREUM CONNECTION

📡 Connecting to Ethereum mainnet via Infura...
   API Key: 910cab7e...dbd36b1d

✅ Successfully connected to Ethereum mainnet!

📦 Latest block number: 24,604,202
⏰ Latest block timestamp: 2026-03-07 16:52:23

⛽ Current gas price: 0.05 Gwei
💸 Cost of ETH transfer: 0.000001 ETH
   (21,000 gas × 0.05 Gwei)

💱 Cost of Uniswap swap: 0.000007 ETH
   (150,000 gas × 0.05 Gwei)

🌐 Blockchain connection established!
   You can now read any public data from Ethereum
   Including: prices, pool reserves, balances, etc.

WHAT YOU CAN DO NOW:

With this Web3 connection, you can:
  • Read Uniswap pool reserves
  • Query token balances
  • Monitor liquidity changes
  • Track impermanent loss
  • Get live ETH prices from DEX
  • All without paying gas fees!

✅ Ready for Cell 2: Query Uniswap V3 pool for live ETH/USDC data


In [4]:
# ============================================================
# CELL 2: QUERY UNISWAP V3 POOL FOR LIVE ETH PRICE (CORRECTED)
# ============================================================
#
# WHAT IS UNISWAP V3?
# ─────────────────────────────────────────────────────────────
# Uniswap is a Decentralized Exchange (DEX) on Ethereum
# - No central authority (runs on smart contracts)
# - Anyone can provide liquidity and earn fees
# - Pools contain pairs of tokens (e.g., ETH/USDC)
#
# IMPORTANT: Token order matters!
# ─────────────────────────────────────────────────────────────
# Uniswap pools have token0 and token1
# The POOL determines which is which (not us!)
# We need to check which token is which, then adjust our price calculation
# ============================================================

print("=" * 70)
print("QUERYING UNISWAP V3 ETH/USDC POOL")
print("=" * 70)
print()

# ─────────────────────────────────────────────────────────────
# STEP 1: DEFINE KNOWN TOKEN ADDRESSES
# ─────────────────────────────────────────────────────────────
# WETH (Wrapped ETH) and USDC have fixed addresses on Ethereum
# We'll use these to identify which token is which in the pool
# ─────────────────────────────────────────────────────────────

WETH_ADDRESS = '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2'
USDC_ADDRESS = '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48'

# Uniswap V3 ETH/USDC pool address (0.05% fee tier)
POOL_ADDRESS = '0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640'

pool_address = w3.to_checksum_address(POOL_ADDRESS)
weth_address = w3.to_checksum_address(WETH_ADDRESS)
usdc_address = w3.to_checksum_address(USDC_ADDRESS)

print(f"📍 Pool Address: {pool_address}")
print(f"   Pair: ETH/USDC")
print(f"   Fee Tier: 0.05%")
print()

# ─────────────────────────────────────────────────────────────
# STEP 2: LOAD POOL ABI
# ─────────────────────────────────────────────────────────────

POOL_ABI = [
    {
        "inputs": [],
        "name": "slot0",
        "outputs": [
            {"internalType": "uint160", "name": "sqrtPriceX96", "type": "uint160"},
            {"internalType": "int24", "name": "tick", "type": "int24"},
            {"internalType": "uint16", "name": "observationIndex", "type": "uint16"},
            {"internalType": "uint16", "name": "observationCardinality", "type": "uint16"},
            {"internalType": "uint16", "name": "observationCardinalityNext", "type": "uint16"},
            {"internalType": "uint8", "name": "feeProtocol", "type": "uint8"},
            {"internalType": "bool", "name": "unlocked", "type": "bool"}
        ],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "liquidity",
        "outputs": [{"internalType": "uint128", "name": "", "type": "uint128"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "token0",
        "outputs": [{"internalType": "address", "name": "", "type": "address"}],
        "stateMutability": "view",
        "type": "function"
    },
    {
        "inputs": [],
        "name": "token1",
        "outputs": [{"internalType": "address", "name": "", "type": "address"}],
        "stateMutability": "view",
        "type": "function"
    }
]

pool_contract = w3.eth.contract(address=pool_address, abi=POOL_ABI)
print("✅ Pool contract loaded")
print()

# ─────────────────────────────────────────────────────────────
# STEP 3: READ POOL DATA
# ─────────────────────────────────────────────────────────────

slot0 = pool_contract.functions.slot0().call()
sqrtPriceX96 = slot0[0]
tick = slot0[1]
liquidity = pool_contract.functions.liquidity().call()

# Get token addresses and CHECK which is which
token0 = pool_contract.functions.token0().call()
token1 = pool_contract.functions.token1().call()

print("📊 Raw Pool Data:")
print(f"   sqrtPriceX96: {sqrtPriceX96:,}")
print(f"   Current tick: {tick:,}")
print(f"   Liquidity: {liquidity:,}")
print()

# ─────────────────────────────────────────────────────────────
# STEP 4: IDENTIFY TOKEN ORDER
# ─────────────────────────────────────────────────────────────
# Check which token is WETH and which is USDC
# This determines how we calculate the price
# ─────────────────────────────────────────────────────────────

print("🔍 Token Identification:")
print(f"   Token0: {token0}")
print(f"   Token1: {token1}")
print()

# Check if WETH is token0 or token1
if token0.lower() == weth_address.lower():
    print("   ✅ Token0 = WETH (18 decimals)")
    print("   ✅ Token1 = USDC (6 decimals)")
    weth_is_token0 = True
elif token1.lower() == weth_address.lower():
    print("   ✅ Token0 = USDC (6 decimals)")
    print("   ✅ Token1 = WETH (18 decimals)")
    weth_is_token0 = False
else:
    print("   ❌ Error: Neither token is WETH!")
    weth_is_token0 = None

print()

# ─────────────────────────────────────────────────────────────
# STEP 5: CALCULATE PRICE CORRECTLY
# ─────────────────────────────────────────────────────────────
# sqrtPriceX96 represents: sqrt(token1/token0) * 2^96
# 
# If WETH is token0:
#   price = (sqrtPriceX96 / 2^96)^2 * 10^(18-6)
#   This gives WETH price in USDC
#
# If WETH is token1:
#   We need to INVERT the calculation
#   price = 1 / [(sqrtPriceX96 / 2^96)^2] * 10^(6-18)
#   Then adjust: * 10^12 to get WETH price in USDC
# ─────────────────────────────────────────────────────────────

if weth_is_token0:
    # WETH is token0: price = token1/token0 = USDC/WETH
    # This gives us USDC per WETH directly
    price = (sqrtPriceX96 / (2**96)) ** 2 * (10**12)
    print("📐 Calculation: WETH is token0")
    print(f"   Price = (sqrtPriceX96 / 2^96)^2 * 10^12")
    
else:
    # WETH is token1: price = token1/token0 = WETH/USDC
    # This gives us WETH per USDC (inverted!)
    # We need to flip it: USDC per WETH = 1 / (WETH per USDC)
    raw_price = (sqrtPriceX96 / (2**96)) ** 2
    # Invert and adjust for decimals
    # token0 = USDC (6 decimals), token1 = WETH (18 decimals)
    # Decimal adjustment: 10^(token1_decimals - token0_decimals) = 10^12
    price = (1 / raw_price) * (10**12)
    print("📐 Calculation: WETH is token1 (need to invert)")
    print(f"   Raw price (WETH/USDC): {raw_price:.15f}")
    print(f"   Inverted price (USDC/WETH): {price:.2f}")

print()

print("💰 LIVE ETH PRICE FROM UNISWAP V3:")
print(f"   ${price:,.2f} per ETH")
print()

# ─────────────────────────────────────────────────────────────
# STEP 6: SANITY CHECK
# ─────────────────────────────────────────────────────────────
# ETH price should be between $1,000 and $10,000 typically
# If outside this range, something is wrong!
# ─────────────────────────────────────────────────────────────

if 1000 <= price <= 10000:
    print("✅ SANITY CHECK PASSED")
    print(f"   Price ${price:,.2f} is within reasonable range")
elif price < 1000:
    print("⚠️  WARNING: Price seems LOW")
    print(f"   ${price:,.2f} is below typical ETH range")
elif price > 10000:
    print("⚠️  WARNING: Price seems HIGH")
    print(f"   ${price:,.2f} is above typical ETH range")

print()

print("=" * 70)
print("SUCCESS!")
print("=" * 70)
print()
print("You just read LIVE ETH price from Uniswap V3!")
print("This is the ACTUAL on-chain price right now.")
print()
print("✅ Next: Build automated price fetcher function")

QUERYING UNISWAP V3 ETH/USDC POOL

📍 Pool Address: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640
   Pair: ETH/USDC
   Fee Tier: 0.05%

✅ Pool contract loaded

📊 Raw Pool Data:
   sqrtPriceX96: 1,778,250,906,494,042,191,521,234,757,473,912
   Current tick: 200,386
   Liquidity: 14,395,984,821,373,200,053

🔍 Token Identification:
   Token0: 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48
   Token1: 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2

   ✅ Token0 = USDC (6 decimals)
   ✅ Token1 = WETH (18 decimals)

📐 Calculation: WETH is token1 (need to invert)
   Raw price (WETH/USDC): 503763746.351338505744934
   Inverted price (USDC/WETH): 1985.06

💰 LIVE ETH PRICE FROM UNISWAP V3:
   $1,985.06 per ETH

✅ SANITY CHECK PASSED
   Price $1,985.06 is within reasonable range

SUCCESS!

You just read LIVE ETH price from Uniswap V3!
This is the ACTUAL on-chain price right now.

✅ Next: Build automated price fetcher function


In [6]:
# ============================================================
# CELL 3: BUILD AUTOMATED PRICE FETCHER FUNCTION
# ============================================================
#
# Now that we know how to read Uniswap prices, let's create
# a reusable function that can fetch prices on demand.
#
# This function will:
# - Automatically detect token order
# - Calculate price correctly
# - Handle errors gracefully
# - Return clean price data
#
# USE CASES:
# ─────────────────────────────────────────────────────────────
# 1. Real-time price monitoring
# 2. Compare CEX (Binance) vs DEX (Uniswap) prices
# 3. Detect arbitrage opportunities
# 4. Integrate DeFi prices into trading strategies
# ============================================================

import time

def get_uniswap_eth_price(w3, pool_address, weth_address, usdc_address):
    """
    Fetch live ETH/USDC price from Uniswap V3 pool
    
    Parameters:
    ─────────────────────────────
    w3 [VARIABLE - Web3] : Web3 connection instance
    pool_address [VARIABLE - str] : Uniswap V3 pool contract address
    weth_address [VARIABLE - str] : WETH token address
    usdc_address [VARIABLE - str] : USDC token address
    
    Returns:
    ─────────────────────────────
    dict : {
        'price': float (ETH price in USDC),
        'sqrtPriceX96': int (raw price from contract),
        'tick': int (current tick),
        'liquidity': int (total liquidity),
        'timestamp': datetime (when price was fetched),
        'block_number': int (block number when fetched)
    }
    
    Returns None if fetch fails
    """
    
    try:
        # Minimal pool ABI
        pool_abi = [
            {
                "inputs": [],
                "name": "slot0",
                "outputs": [
                    {"name": "sqrtPriceX96", "type": "uint160"},
                    {"name": "tick", "type": "int24"},
                    {"name": "observationIndex", "type": "uint16"},
                    {"name": "observationCardinality", "type": "uint16"},
                    {"name": "observationCardinalityNext", "type": "uint16"},
                    {"name": "feeProtocol", "type": "uint8"},
                    {"name": "unlocked", "type": "bool"}
                ],
                "stateMutability": "view",
                "type": "function"
            },
            {
                "inputs": [],
                "name": "liquidity",
                "outputs": [{"name": "", "type": "uint128"}],
                "stateMutability": "view",
                "type": "function"
            },
            {
                "inputs": [],
                "name": "token0",
                "outputs": [{"name": "", "type": "address"}],
                "stateMutability": "view",
                "type": "function"
            },
            {
                "inputs": [],
                "name": "token1",
                "outputs": [{"name": "", "type": "address"}],
                "stateMutability": "view",
                "type": "function"
            }
        ]
        
        # Create contract instance
        pool = w3.eth.contract(
            address=w3.to_checksum_address(pool_address),
            abi=pool_abi
        )
        
        # Read pool state
        slot0 = pool.functions.slot0().call()
        sqrtPriceX96 = slot0[0]
        tick = slot0[1]
        liquidity = pool.functions.liquidity().call()
        
        # Get token order
        token0 = pool.functions.token0().call()
        token1 = pool.functions.token1().call()
        
        # Determine if WETH is token0 or token1
        weth_is_token0 = (token0.lower() == weth_address.lower())
        
        # Calculate price based on token order
        if weth_is_token0:
            # WETH is token0: price = token1/token0
            price = (sqrtPriceX96 / (2**96)) ** 2 * (10**12)
        else:
            # WETH is token1: need to invert
            raw_price = (sqrtPriceX96 / (2**96)) ** 2
            price = (1 / raw_price) * (10**12)
        
        # Get current block info
        block_number = w3.eth.block_number
        block = w3.eth.get_block(block_number)
        timestamp = datetime.fromtimestamp(block['timestamp'])
        
        return {
            'price': price,
            'sqrtPriceX96': sqrtPriceX96,
            'tick': tick,
            'liquidity': liquidity,
            'timestamp': timestamp,
            'block_number': block_number
        }
        
    except Exception as e:
        print(f"❌ Error fetching price: {e}")
        return None


# ─────────────────────────────────────────────────────────────
# TEST THE FUNCTION
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("TESTING AUTOMATED PRICE FETCHER")
print("=" * 70)
print()

# Fetch price using our new function
result = get_uniswap_eth_price(w3, POOL_ADDRESS, WETH_ADDRESS, USDC_ADDRESS)

if result:
    print("✅ Price fetch successful!")
    print()
    print(f"💰 ETH Price: ${result['price']:,.2f}")
    print(f"📦 Block: {result['block_number']:,}")
    print(f"⏰ Time: {result['timestamp']}")
    print(f"📊 Liquidity: {result['liquidity']:,}")
    print(f"🎯 Tick: {result['tick']:,}")
    print()
else:
    print("❌ Price fetch failed!")
    print()

# ─────────────────────────────────────────────────────────────
# FETCH MULTIPLE PRICES (SIMULATE REAL-TIME MONITORING)
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("REAL-TIME PRICE MONITORING (5 samples)")
print("=" * 70)
print()

prices = []

for i in range(5):
    result = get_uniswap_eth_price(w3, POOL_ADDRESS, WETH_ADDRESS, USDC_ADDRESS)
    
    if result:
        prices.append(result)
        print(f"Sample {i+1}: ${result['price']:,.2f} at block {result['block_number']:,}")
        
    # Wait 3 seconds between samples
    # (Ethereum blocks every ~12 seconds, but price can change within a block)
    if i < 4:  # Don't sleep after last iteration
        time.sleep(3)

print()

# ─────────────────────────────────────────────────────────────
# ANALYZE PRICE STABILITY
# ─────────────────────────────────────────────────────────────

if len(prices) > 0:
    price_values = [p['price'] for p in prices]
    
    avg_price = sum(price_values) / len(price_values)
    min_price = min(price_values)
    max_price = max(price_values)
    price_range = max_price - min_price
    
    print("📊 PRICE ANALYSIS (over 15 seconds):")
    print(f"   Average: ${avg_price:,.2f}")
    print(f"   Min:     ${min_price:,.2f}")
    print(f"   Max:     ${max_price:,.2f}")
    print(f"   Range:   ${price_range:,.2f}")
    print(f"   Volatility: {(price_range/avg_price)*100:.4f}%")
    print()

print("=" * 70)
print("SUCCESS!")
print("=" * 70)
print()
print("You now have a reusable function to fetch live ETH prices!")
print("This can be integrated with your ADX regime strategy.")
print()
print("Next steps:")
print("  • Compare Uniswap (DEX) vs Binance (CEX) prices")
print("  • Calculate impermanent loss for LP positions")
print("  • Integrate with ADX regime detection")

TESTING AUTOMATED PRICE FETCHER

✅ Price fetch successful!

💰 ETH Price: $1,986.28
📦 Block: 24,604,359
⏰ Time: 2026-03-07 17:23:59
📊 Liquidity: 14,395,985,781,412,234,784
🎯 Tick: 200,380

REAL-TIME PRICE MONITORING (5 samples)

Sample 1: $1,986.28 at block 24,604,359
Sample 2: $1,986.28 at block 24,604,360
Sample 3: $1,986.28 at block 24,604,360
Sample 4: $1,986.28 at block 24,604,361
Sample 5: $1,986.48 at block 24,604,361

📊 PRICE ANALYSIS (over 15 seconds):
   Average: $1,986.32
   Min:     $1,986.28
   Max:     $1,986.48
   Range:   $0.20
   Volatility: 0.0103%

SUCCESS!

You now have a reusable function to fetch live ETH prices!
This can be integrated with your ADX regime strategy.

Next steps:
  • Compare Uniswap (DEX) vs Binance (CEX) prices
  • Calculate impermanent loss for LP positions
  • Integrate with ADX regime detection
